# **Gerando respostas com uma LLM**

### **Carregando o modelo Llama**

Para que seja possível utilizar uma LLM localmente, precisamos carregar um modelo mais leve, caso contrário o computador não conseguirá processar os resultados.

O [Unsloth](https://unsloth.ai/) fornece LLMs de código aberto e opções quantizadas dos modelos que reduz a memória necessária para o carregamento e melhora a velocidade de processamento:

- [Modelos de código aberto](https://huggingface.co/unsloth)

Vamos instalar a biblioteca Unsloth e pré-requisitos para carregar um modelo Llama. Precisamos utilizar uma GPU para utilização da biblioteca, portanto vamos usar a GPU T4 do Google Colab.

In [1]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/19

- Link git Unsloth: `'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'`



In [2]:
!pip install --upgrade --no-deps 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-jbr1nvui/unsloth_fd790d4b8c6a45878a9973196a082580
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-jbr1nvui/unsloth_fd790d4b8c6a45878a9973196a082580
  Resolved https://github.com/unslothai/unsloth.git to commit 8d39ee11db697a85c28d7432f53072814987c884
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2026.9.5-py3-none-any.whl size=8229257 sha256=d373cbe1d76cb55eb753972edea8ad8eaf9ca1dd2234d1ffff2a6eeaa7a613b2
  Stored in directory: /tmp/pip-ephem-wheel-cache-tb3d_e8n/wheels/d5/36/1d/4e65996c5b80c84a5ac1b0ba10718bdc155f8dd04352746a8f
Successfully built unsloth
  Attempting uninstall: unsloth
    Found existing installation: unsloth 2026.9.4
    Uninstalling unsloth-2026.9.4:
      Successfully uninstalled unsloth-2026.9.4


In [3]:
!pip install  --no-deps torch xformers trl peft accelerate bitsandbytes triton

In [4]:
from unsloth import FastLanguageModel
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Vamos utilizar o modelo LLama 3.1 com 8 bilhões de parâmetros. É um modelo de código aberto, por conta disso não precisamos de acessar nenhuma API, nem pagar nenhum valor para utilizar:

- [Llama 3.1-8B Hugging Face](https://huggingface.co/unsloth/Meta-Llama-3.1-8B)

In [5]:
checkpoint_modelo = 'unsloth/Meta-Llama-3.1-8B'

No momento de fazer o carregamento do modelo, vamos utilizar parâmetros para utilizar menos memória.

- dtype: None para detecção automática, Float16 para Tesla T4, V100, Bfloat16 para Ampere+
- load_in_4bit: Utiliza menos memória ao reduzir a quantidade de bits de informação. Menos preciso.

In [6]:
modelo, tokenizador = FastLanguageModel.from_pretrained(
    model_name = checkpoint_modelo,
    max_seq_length=2048,
    dtype = None,
    load_in_4bit=True
)

==((====))==  Unsloth 2026.9.5: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-unsloth-bnb-4bit as a legacy tokenizer.


In [ ]:
modelo

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaExtendedRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): L

In [ ]:
tokenizador

PreTrainedTokenizerFast(name_or_path='unsloth/meta-llama-3.1-8b-bnb-4bit', vocab_size=128000, model_max_length=131072, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<|begin_of_text|>', 'eos_token': '<|end_of_text|>', 'pad_token': '<|finetune_right_pad_id|>'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	128000: AddedToken("<|begin_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128001: AddedToken("<|end_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128002: AddedToken("<|reserved_special_token_0|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128003: AddedToken("<|reserved_special_token_1|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128004: AddedToken("<|finetune_right_pad_id|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	12

### **Gerando consultas com o modelo**

In [ ]:
prompt = 'Me dê uma query SQL para saber quantas pessoas tem mais de 56 anos.'

In [ ]:
prompt_tokenizado = tokenizador([prompt], return_tensors='pt').to('cuda')

In [ ]:
prompt_tokenizado

{'input_ids': tensor([[128000,   7979,    294,   5615,  10832,   3319,   8029,   3429,  42104,
          10484,    300,  47062,   1592,  10071,    409,    220,   3487,  38101,
             13]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
       device='cuda:0')}

In [ ]:
from transformers import TextStreamer

In [ ]:
FastLanguageModel.for_inference(modelo)
streamer_texto = TextStreamer(tokenizador)

_ = modelo.generate(**prompt_tokenizado, streamer = streamer_texto, max_new_tokens = 128)

<|begin_of_text|>Me dê uma query SQL para saber quantas pessoas tem mais de 56 

Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


anos.<|end_of_text|>


O modelo pode gerar um texto inesperado e alterar o objetivo da pergunta inicial, tirando todo o propósito de utilizar o modelo para gerar a query correta.

In [7]:
from datasets import load_dataset

In [8]:
dataset = load_dataset('emdemor/sql-create-context-pt', split = 'train')

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

sql-pt.parquet: reconstructing file:   0%|          |  0.00B / 6.61MB            

sql-pt.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

In [10]:
dataset.to_pandas()

,pergunta,contexto,resposta
0,Quantos chefes de departamento têm mais de 56 ...,CREATE TABLE head (age INTEGER),SELECT COUNT(*) FROM head WHERE age > 56
1,"Indicar o nome, estado de nascimento e idade d...","CREATE TABLE head (name VARCHAR, born_state VA...","SELECT name, born_state, age FROM head ORDER B..."
2,"Indique o ano de criação, o nome e o orçamento...","CREATE TABLE department (creation VARCHAR, nam...","SELECT creation, name, budget_in_billions FROM..."
3,Qual é o orçamento máximo e mínimo dos departa...,CREATE TABLE department (budget_in_billions IN...,"SELECT MAX(budget_in_billions), MIN(budget_in_..."
4,Qual é o número médio de empregados dos depart...,CREATE TABLE department (num_employees INTEGER...,SELECT AVG(num_employees) FROM department WHER...
...,...,...,...
78572,A que horas foi o jogo com a pontuação de 3-2?,"CREATE TABLE table_name_35 (time VARCHAR, scor...","SELECT time FROM table_name_35 WHERE score = ""..."
78573,Em que terreno a equipa jogou contra o Aston V...,"CREATE TABLE table_name_83 (ground VARCHAR, op...",SELECT ground FROM table_name_83 WHERE opponen...
78574,Que tipo de competição foi no San Siro às 18h3...,CREATE TABLE table_name_60 (competition VARCHA...,SELECT competition FROM table_name_60 WHERE gr...
78575,Qual é o número total de decílios para a local...,"CREATE TABLE table_name_34 (decile VARCHAR, na...",SELECT COUNT(decile) FROM table_name_34 WHERE ...


In [9]:
def gerar_prompt_sql(contexto, pergunta, resposta = ''):
    return f'''Você é um modelo poderoso de texto-para-SQL. Seu trabalho é responder perguntas sobre um banco de dados. Você recebe uma pergunta e o contexto relacionado a uma ou mais tabelas.

Você deve gerar a consulta SQL que responde à pergunta.

### Instruction:
Contexto: {contexto}

### Input:
Pergunta: {pergunta}

### Response:
Resposta: {resposta}
'''

In [ ]:
dataset[0]

{'pergunta': 'Quantos chefes de departamento têm mais de 56 anos ?',
 'contexto': 'CREATE TABLE head (age INTEGER)',
 'resposta': 'SELECT COUNT(*) FROM head WHERE age > 56'}

In [ ]:
print(gerar_prompt_sql(dataset[0]['contexto'], dataset[0]['pergunta'], dataset[0]['resposta']))

Você é um modelo poderoso de texto-para-SQL. Seu trabalho é responder perguntas sobre um banco de dados. Você recebe uma pergunta e o contexto relacionado a uma ou mais tabelas.

Você deve gerar a consulta SQL que responde à pergunta.

### Instruction:
Contexto: CREATE TABLE head (age INTEGER)

### Input:
Pergunta: Quantos chefes de departamento têm mais de 56 anos ?

### Response:
Resposta: SELECT COUNT(*) FROM head WHERE age > 56



In [11]:
EOS_TOKEN = tokenizador.eos_token

In [12]:
EOS_TOKEN

'<|end_of_text|>'

In [13]:
def formatar_prompts(dados):
    contextos = dados['contexto']
    perguntas = dados['pergunta']
    respostas = dados['resposta']
    textos = []
    for contexto, pergunta, resposta in zip(contextos, perguntas, respostas):
        texto = gerar_prompt_sql(contexto, pergunta, resposta) + EOS_TOKEN
        textos.append(texto)
    return {'texto': textos,}

In [14]:
dataset = dataset.map(formatar_prompts, batched = True)

Map:   0%|          | 0/78577 [00:00<?, ? examples/s]

In [ ]:
dataset.to_pandas()

,pergunta,contexto,resposta,texto
0,Quantos chefes de departamento têm mais de 56 ...,CREATE TABLE head (age INTEGER),SELECT COUNT(*) FROM head WHERE age > 56,Você é um modelo poderoso de texto-para-SQL. S...
1,"Indicar o nome, estado de nascimento e idade d...","CREATE TABLE head (name VARCHAR, born_state VA...","SELECT name, born_state, age FROM head ORDER B...",Você é um modelo poderoso de texto-para-SQL. S...
2,"Indique o ano de criação, o nome e o orçamento...","CREATE TABLE department (creation VARCHAR, nam...","SELECT creation, name, budget_in_billions FROM...",Você é um modelo poderoso de texto-para-SQL. S...
3,Qual é o orçamento máximo e mínimo dos departa...,CREATE TABLE department (budget_in_billions IN...,"SELECT MAX(budget_in_billions), MIN(budget_in_...",Você é um modelo poderoso de texto-para-SQL. S...
4,Qual é o número médio de empregados dos depart...,CREATE TABLE department (num_employees INTEGER...,SELECT AVG(num_employees) FROM department WHER...,Você é um modelo poderoso de texto-para-SQL. S...
...,...,...,...,...
78572,A que horas foi o jogo com a pontuação de 3-2?,"CREATE TABLE table_name_35 (time VARCHAR, scor...","SELECT time FROM table_name_35 WHERE score = ""...",Você é um modelo poderoso de texto-para-SQL. S...
78573,Em que terreno a equipa jogou contra o Aston V...,"CREATE TABLE table_name_83 (ground VARCHAR, op...",SELECT ground FROM table_name_83 WHERE opponen...,Você é um modelo poderoso de texto-para-SQL. S...
78574,Que tipo de competição foi no San Siro às 18h3...,CREATE TABLE table_name_60 (competition VARCHA...,SELECT competition FROM table_name_60 WHERE gr...,Você é um modelo poderoso de texto-para-SQL. S...
78575,Qual é o número total de decílios para a local...,"CREATE TABLE table_name_34 (decile VARCHAR, na...",SELECT COUNT(decile) FROM table_name_34 WHERE ...,Você é um modelo poderoso de texto-para-SQL. S...


In [ ]:
dataset['texto']

Column(['Você é um modelo poderoso de texto-para-SQL. Seu trabalho é responder perguntas sobre um banco de dados. Você recebe uma pergunta e o contexto relacionado a uma ou mais tabelas.\n\nVocê deve gerar a consulta SQL que responde à pergunta.\n\n### Instruction:\nContexto: CREATE TABLE head (age INTEGER)\n\n### Input:\nPergunta: Quantos chefes de departamento têm mais de 56 anos ?\n\n### Response:\nResposta: SELECT COUNT(*) FROM head WHERE age > 56\n<|end_of_text|>', 'Você é um modelo poderoso de texto-para-SQL. Seu trabalho é responder perguntas sobre um banco de dados. Você recebe uma pergunta e o contexto relacionado a uma ou mais tabelas.\n\nVocê deve gerar a consulta SQL que responde à pergunta.\n\n### Instruction:\nContexto: CREATE TABLE head (name VARCHAR, born_state VARCHAR, age VARCHAR)\n\n### Input:\nPergunta: Indicar o nome, estado de nascimento e idade dos chefes de departamento, ordenados por idade.\n\n### Response:\nResposta: SELECT name, born_state, age FROM head ORDE

In [15]:
modelo = FastLanguageModel.get_peft_model(
    modelo,
    r = 16,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state = 10,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2026.9.5 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [16]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

In [17]:

def formatting_func_for_trainer(example):
    return [str(example["texto"])]


trainer = SFTTrainer(
    model=modelo,
    tokenizer=tokenizador,
    train_dataset=dataset,
    formatting_func=formatting_func_for_trainer,
    max_seq_length=2048,
    dataset_num_proc=1,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        learning_rate=2e-5,
        max_steps=60,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=10,
        output_dir="outputs",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/78577 [00:00<?, ? examples/s]

In [18]:
from transformers import TextStreamer

FastLanguageModel.for_inference(modelo)

prompt = "Me dê uma query SQL para saber quantas pessoas tem mais de 56 anos."

inputs = tokenizador(
    prompt,
    return_tensors="pt"
).to("cuda")

streamer_texto = TextStreamer(
    tokenizador,
    skip_prompt=True
)

_ = modelo.generate(
    **inputs,
    streamer=streamer_texto,
    max_new_tokens=32
)

Both `max_new_tokens` (=32) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|end_of_text|>


In [19]:
from huggingface_hub import notebook_login

notebook_login()


In [20]:
modelo.push_to_hub_gguf('llama-3.1-8B-texto-para-sql',tokenizador,quantization_method='q4_k_m')

Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/947 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /tmp/unsloth_gguf_eo214kd4/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...-llama-3.1-8b.Q4_K_M.gguf:   0%|          | 14.0MB / 4.92GB            

Uploading config.json...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/fabricioribeiro/llama-3.1-8B-texto-para-sql
Unsloth: Cleaning up temporary files...


'fabricioribeiro/llama-3.1-8B-texto-para-sql'